In [1]:
%pip install -q pandas

Note: you may need to restart the kernel to use updated packages.


## Könyvek feldarabolása

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
CHUNKS_DIR = Path("chunks")
LANGS = ["en", "hu"]

CHUNK_WORDS = 200
OVERLAP_WORDS = 40

CHUNKS_DIR.mkdir(exist_ok=True)
for lang in LANGS:
    (DATA_DIR / lang).mkdir(parents=True, exist_ok=True)

## A könyvek beolvasása és tisztítása

In [ ]:
import re, unicodedata

CHAR_MAP = {
    "‘": "'", "’": "'", "‚": "'", "‛": "'",
    "“": '"', "”": '"', "„": '"', "»": '"', "«": '"',
    "–": "-", "—": "-", "−": "-", " ": " ", "­": "",
    "…": "...",
}
_CHAR_RE = re.compile("|".join(map(re.escape, CHAR_MAP)))

def clean_text(text: str) -> str:
    """Unify quotes/dashes, re-join hyphenated line breaks, collapse whitespace (keeps case)."""
    s = unicodedata.normalize("NFC", text)
    s = _CHAR_RE.sub(lambda m: CHAR_MAP[m.group(0)], s)
    s = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def normalize(text: str) -> str:
    """For matching: cleaned + lowercase."""
    return clean_text(str(text)).lower()


def read_txt(path):
    for enc in ("utf-8", "utf-8-sig", "cp1250", "iso-8859-2"):
        try:
            return path.read_text(encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Cannot decode: {path}")

CORPUS = {}
for lang in LANGS:
    files = sorted((DATA_DIR / lang).glob("*.txt"))
    if not files:
        print(f"WARNING: no .txt files in {DATA_DIR / lang}")
    CORPUS[lang] = {f.stem: clean_text(read_txt(f)) for f in files}
    for book, text in CORPUS[lang].items():
        print(f"[{lang}] {book}: {len(text.split()):,} words")

[en] 1_Hobbit_or_there_and_back_again: 95,910 words
[en] 2_The_Fellowship_Of_The_Ring: 188,432 words
[en] 3_The_Two_Towers: 154,616 words
[en] 4_The_Return_Of_The_King: 135,443 words
[hu] 1_a_hobbit: 75,859 words
[hu] 2_a_gyuruk_ura_a_gyuru_szovetsege: 155,818 words
[hu] 3_a_gyuruk_ura_a_kiraly_visszater: 108,664 words
[hu] 4_a_gyuruk_ura_a_ket_torony: 126,751 words


## Darabolás

In [4]:
import pandas as pd, json

def chunk_text(text, size=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    words = text.split()
    step = size - overlap
    assert step > 0, "Overlap must be smaller than chunk size."
    return [(start, " ".join(words[start:start + size]))
            for start in range(0, max(len(words) - overlap, 1), step)]

for lang in LANGS:
    rows = []
    for book, text in CORPUS[lang].items():
        for i, (start, chunk) in enumerate(chunk_text(text)):
            rows.append({"id": f"{lang}_{book}_{i:05d}", "book": book, "position": i,
                         "start_word": start, "n_words": len(chunk.split()), "text": chunk})
    df = pd.DataFrame(rows)
    with open(CHUNKS_DIR / f"{lang}.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    df.to_csv(CHUNKS_DIR / f"{lang}.csv", index=False, encoding="utf-8-sig")
    print(f"[{lang}] {len(df):,} chunks, mean {df['n_words'].mean():.0f} words -> {CHUNKS_DIR / (lang + '.jsonl')}")

_ = (CHUNKS_DIR / "info.json").write_text(json.dumps(
    {"method": "fixed size + overlap (words)", "chunk_words": CHUNK_WORDS, "overlap_words": OVERLAP_WORDS,
     "books": {lang: list(CORPUS[lang]) for lang in LANGS}}, ensure_ascii=False, indent=2), encoding="utf-8")

[en] 3,592 chunks, mean 200 words -> chunks\en.jsonl
[hu] 2,919 chunks, mean 200 words -> chunks\hu.jsonl


## Ellenőrzés

In [6]:
for lang in LANGS:
    df = pd.read_json(CHUNKS_DIR / f"{lang}.jsonl", lines=True)
    for _, r in df.head(2).iterrows():
        print(f"{r['id']} ({r['n_words']} words)\n   {r['text'][:300]} ...\n")

en_1_Hobbit_or_there_and_back_again_00000 (200 words)
   CHAPTER ONE An Unexpected Party In a hole in the ground there lived a hobbit. Not a nasty, dirty, wet hole, filled with the ends of worms and an oozy smell, nor yet a dry, bare, sandy hole with nothing in it to sit down on or to eat: it was a hobbit-hole, and that means comfort. It had a perfectly r ...

en_1_Hobbit_or_there_and_back_again_00001 (200 words)
   doors opened out of it, first on one side and then on another. No going upstairs for the hobbit: bedrooms, bathrooms, cellars, pantries (lots of these), wardrobes (he had whole rooms devoted to clothes), kitchens, dining-rooms, all were on the same floor, and indeed on the same passage. The best roo ...

hu_1_a_hobbit_00000 (200 words)
   J.R.R. Tolkien A Hobbit 1. fejezet Egy váratlan összejövetel Egy földbevájt üregben élt egy hobbit. Nem valami mocskos, koszos, nedves üregben, ami férgek maradványaival és nyirkos bűzzel van tele, és nem is afféle száraz, csupasz, homokos